# AetherForge · Enterprise Training Dashboard
Palantir-grade monitoring for every training run. Works in Jupyter, JupyterLab, VS Code, and Google Colab.

In [ ]:
# ── 0. Dependencies ──────────────────────────────────────────────────────
import importlib, subprocess, sys

def ensure(*pkgs):
    for pkg in pkgs:
        mod = pkg.split('[')[0].replace('-','_')
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

ensure('pandas>=2.0', 'plotly', 'scipy')
print('Dependencies ready.')

In [ ]:
# ── 1. Configuration — change paths here ─────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from pathlib import Path
import datetime

# ── Point to whichever run you want to inspect ───────────────────────────
RUNS = {
    'Qwen2.5-VL LoRA (text)':     './outputs/qwen25_vl_lora/training_log.csv',
    'AetherForge 128M pretrain':   './outputs/aetherforge_pretrain/training_log.csv',
}

# Set active run (key from RUNS dict, or a direct path string)
ACTIVE_RUN  = 'Qwen2.5-VL LoRA (text)'
MA_WINDOW   = 10     # moving-average window for loss smoothing
VRAM_BUDGET = 16.0   # GB — your RTX 4080 Super; shown as a budget line

# ── Resolve path ─────────────────────────────────────────────────────────
log_path = Path(RUNS.get(ACTIVE_RUN, ACTIVE_RUN))
if not log_path.exists():
    # Fall back to whichever run exists
    for name, path in RUNS.items():
        if Path(path).exists():
            log_path   = Path(path)
            ACTIVE_RUN = name
            print(f'Primary path not found — falling back to: {name}')
            break
    else:
        raise FileNotFoundError(
            'No training_log.csv found.\n'
            'Run: make finetune-test   OR   make train-aetherforge-test'
        )

df = pd.read_csv(log_path)
print(f'Loaded {len(df)} steps  ·  {ACTIVE_RUN}  ·  {log_path}')
df.head(3)

In [ ]:
# ── 2. Palantir design tokens ─────────────────────────────────────────────

# Colour palette
C = dict(
    bg         = '#0D1117',   # page background
    surface    = '#161B22',   # card surface
    surface2   = '#21262D',   # nested surface / table header
    border     = '#30363D',
    text       = '#E6EDF3',
    text_dim   = '#7D8590',
    gold       = '#E3B341',   # primary accent (Palantir amber)
    blue       = '#58A6FF',   # secondary accent
    green      = '#3FB950',
    red        = '#F85149',
    purple     = '#BC8CFF',
    orange     = '#F0883E',
)

# Shared Plotly layout kwargs
DARK_LAYOUT = dict(
    paper_bgcolor = C['surface'],
    plot_bgcolor  = C['bg'],
    font          = dict(family='Inter, Segoe UI, system-ui, sans-serif',
                         color=C['text'], size=12),
    xaxis = dict(gridcolor=C['border'], zerolinecolor=C['border'],
                 tickfont=dict(color=C['text_dim'])),
    yaxis = dict(gridcolor=C['border'], zerolinecolor=C['border'],
                 tickfont=dict(color=C['text_dim'])),
    legend = dict(bgcolor='rgba(0,0,0,0)', bordercolor=C['border'],
                  borderwidth=1, font=dict(color=C['text'])),
    margin = dict(l=56, r=24, t=56, b=48),
)

def dark(fig, title='', height=380):
    """Apply enterprise dark theme to a figure."""
    kw = dict(**DARK_LAYOUT,
              title=dict(text=title, font=dict(size=15, color=C['text']),
                         x=0.02, xanchor='left'),
              height=height)
    fig.update_layout(**kw)
    for ax in ['xaxis','xaxis2','xaxis3','xaxis4']:
        fig.update_layout(**{ax: dict(gridcolor=C['border'], zerolinecolor=C['border'])})
    for ax in ['yaxis','yaxis2','yaxis3','yaxis4']:
        fig.update_layout(**{ax: dict(gridcolor=C['border'], zerolinecolor=C['border'])})
    return fig

print('Design tokens loaded.')

In [ ]:
# ── 3. Compute derived metrics ────────────────────────────────────────────
from scipy.signal import savgol_filter

df['loss_ma']  = df['loss'].rolling(MA_WINDOW, min_periods=1).mean()
df['loss_std'] = df['loss'].rolling(MA_WINDOW, min_periods=1).std().fillna(0)

# Smoothed with Savitzky-Golay for a clean trend line
wl = min(len(df) if len(df) % 2 == 1 else len(df) - 1, 11)
if len(df) > 4:
    df['loss_sg'] = savgol_filter(df['loss'], window_length=wl, polyorder=2)
else:
    df['loss_sg'] = df['loss_ma']

total_sec  = df['elapsed_sec'].max()
best_step  = df.loc[df['loss'].idxmin(), 'step']
best_loss  = df['loss'].min()
final_loss = df['loss'].iloc[-1]
pct_drop   = (df['loss'].iloc[0] - final_loss) / df['loss'].iloc[0] * 100

print(f'Best loss {best_loss:.4f} at step {best_step}  ·  '
      f'Final loss {final_loss:.4f}  ·  '
      f'Drop {pct_drop:.1f}%')

In [ ]:
# ── 4. Enterprise Header ──────────────────────────────────────────────────
run_date = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
total_steps_logged = len(df)

def kpi(value, label, color, delta=None):
    delta_html = ''
    if delta is not None:
        arrow   = '▲' if delta >= 0 else '▼'
        d_color = C['red'] if delta >= 0 else C['green']
        delta_html = f'<span style="font-size:11px;color:{d_color}">{arrow} {abs(delta):.1f}%</span>'
    return f'''
      <div style="background:{C["surface2"]};border:1px solid {C["border"]};
                  border-radius:6px;padding:18px 22px;flex:1;min-width:160px;">
        <div style="font-size:11px;font-weight:600;letter-spacing:0.08em;
                    color:{C["text_dim"]};text-transform:uppercase;margin-bottom:8px;">{label}</div>
        <div style="font-size:26px;font-weight:700;color:{color};letter-spacing:-0.5px;
                    font-family:'JetBrains Mono',monospace;">{value}</div>
        <div style="margin-top:4px;">{delta_html}</div>
      </div>'''

cards = ''.join([
    kpi(f'{final_loss:.4f}',  'Final Loss',         C['gold'],   delta=pct_drop),
    kpi(f'{best_loss:.4f}',   'Best Loss',          C['blue']),
    kpi(f"{df['vram_gb'].max():.1f} GB",  'Peak VRAM',  C['green']),
    kpi(f"{int(total_sec//60)}m {int(total_sec%60)}s", 'Train Time', C['purple']),
    kpi(f"{df['tokens_per_sec'].mean():.0f}", 'Avg tok/s',  C['orange']),
    kpi(f'{total_steps_logged}', 'Steps Logged', C['text']),
])

header = f'''
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&family=JetBrains+Mono:wght@500&display=swap');
  .aether-db {{ font-family: Inter, system-ui, sans-serif; }}
</style>
<div class="aether-db" style="background:{C["bg"]};border:1px solid {C["border"]};
     border-radius:8px;padding:24px 28px 20px;margin-bottom:8px;">
  <!-- Title row -->
  <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:20px;">
    <div>
      <div style="font-size:9px;font-weight:700;letter-spacing:0.18em;
                  color:{C["gold"]};text-transform:uppercase;margin-bottom:4px;
                  font-family:'JetBrains Mono',monospace;">AetherForge · Enterprise Intelligence Platform</div>
      <div style="font-size:22px;font-weight:700;color:{C["text"]};letter-spacing:-0.3px;">Training Run Monitor</div>
      <div style="font-size:13px;color:{C["text_dim"]};margin-top:4px;">{ACTIVE_RUN}</div>
    </div>
    <div style="text-align:right;">
      <div style="display:inline-block;background:{C["surface2"]};border:1px solid {C["border"]};
           border-radius:4px;padding:6px 14px;font-size:12px;color:{C["text_dim"]};
           font-family:'JetBrains Mono',monospace;">{run_date}</div>
      <div style="margin-top:8px;">
        <span style="display:inline-block;background:{C["green"]};width:7px;height:7px;
               border-radius:50%;margin-right:6px;"></span>
        <span style="font-size:11px;color:{C["text_dim"]}">Log loaded · {total_steps_logged} entries</span>
      </div>
    </div>
  </div>
  <!-- KPI strip -->
  <div style="display:flex;gap:10px;flex-wrap:wrap;">{cards}</div>
</div>
'''
display(HTML(header))

In [ ]:
# ── 5. Loss curve — main panel ────────────────────────────────────────────
fig = go.Figure()

# Confidence band (±1 std)
fig.add_trace(go.Scatter(
    x=pd.concat([df['step'], df['step'][::-1]]),
    y=pd.concat([df['loss_ma'] + df['loss_std'],
                 (df['loss_ma'] - df['loss_std'])[::-1]]),
    fill='toself',
    fillcolor='rgba(88,166,255,0.07)',
    line=dict(color='rgba(0,0,0,0)'),
    hoverinfo='skip',
    showlegend=False,
    name='±1σ band',
))

# Raw loss (faint)
fig.add_trace(go.Scatter(
    x=df['step'], y=df['loss'],
    name='Raw loss', mode='lines',
    line=dict(color='rgba(88,166,255,0.25)', width=1),
))

# Savitzky-Golay trend
fig.add_trace(go.Scatter(
    x=df['step'], y=df['loss_sg'],
    name='Trend (S-G)', mode='lines',
    line=dict(color=C['blue'], width=2.5),
))

# MA line
fig.add_trace(go.Scatter(
    x=df['step'], y=df['loss_ma'],
    name=f'MA-{MA_WINDOW}', mode='lines',
    line=dict(color=C['gold'], width=1.5, dash='dot'),
))

# Best-loss marker
fig.add_trace(go.Scatter(
    x=[best_step], y=[best_loss],
    mode='markers+text',
    marker=dict(color=C['green'], size=10, symbol='diamond',
                line=dict(color=C['bg'], width=2)),
    text=[f'  Best: {best_loss:.4f}'],
    textposition='middle right',
    textfont=dict(color=C['green'], size=11),
    name='Best checkpoint',
    showlegend=True,
))

dark(fig, title='Loss · Training Dynamics', height=420)
fig.update_layout(
    xaxis_title='Optimization Step',
    yaxis_title='Cross-Entropy Loss',
    hovermode='x unified',
    legend=dict(x=0.72, y=0.98, bgcolor='rgba(22,27,34,0.8)',
                bordercolor=C['border'], borderwidth=1),
)
fig.show()

In [ ]:
# ── 6. LR schedule + VRAM (2-panel row) ──────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Learning Rate Schedule', 'GPU Memory Utilisation'],
    horizontal_spacing=0.09,
)

# LR
fig.add_trace(go.Scatter(
    x=df['step'], y=df['lr'],
    name='LR', mode='lines',
    line=dict(color=C['gold'], width=2.5),
    fill='tozeroy', fillcolor='rgba(227,179,65,0.06)',
), row=1, col=1)

# VRAM with budget line
fig.add_trace(go.Scatter(
    x=df['step'], y=df['vram_gb'],
    name='VRAM used', mode='lines',
    line=dict(color=C['green'], width=2.5),
    fill='tozeroy', fillcolor='rgba(63,185,80,0.07)',
), row=1, col=2)

fig.add_hline(
    y=VRAM_BUDGET, row=1, col=2,
    line=dict(color=C['red'], width=1, dash='dash'),
    annotation_text=f'{VRAM_BUDGET:.0f} GB budget',
    annotation_font_color=C['red'],
    annotation_font_size=10,
)

dark(fig, height=360)
fig.update_xaxes(title_text='Step', title_font_color=C['text_dim'])
fig.update_yaxes(title_text='Learning Rate', row=1, col=1, title_font_color=C['text_dim'])
fig.update_yaxes(title_text='VRAM (GB)',     row=1, col=2, title_font_color=C['text_dim'])
fig.update_annotations(font_size=12, font_color=C['text_dim'])
fig.show()

In [ ]:
# ── 7. Throughput + loss distribution (2-panel row) ───────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Throughput (tokens / sec)', 'Loss Distribution'],
    horizontal_spacing=0.09,
)

# Throughput bars
fig.add_trace(go.Bar(
    x=df['step'], y=df['tokens_per_sec'],
    name='tok/s',
    marker=dict(
        color=df['tokens_per_sec'],
        colorscale=[[0, C['surface2']], [1, C['orange']]],
        showscale=False,
    ),
    opacity=0.85,
), row=1, col=1)

# Rolling mean overlay
fig.add_trace(go.Scatter(
    x=df['step'], y=df['tokens_per_sec'].rolling(5, min_periods=1).mean(),
    mode='lines', name='tok/s MA-5',
    line=dict(color=C['gold'], width=2),
    showlegend=False,
), row=1, col=1)

# Loss histogram
fig.add_trace(go.Histogram(
    x=df['loss'],
    nbinsx=min(30, len(df)),
    name='Loss bins',
    marker=dict(color=C['blue'], opacity=0.75,
                line=dict(color=C['border'], width=0.5)),
), row=1, col=2)

# Vertical lines for mean and best
for val, lbl, col in [
    (df['loss'].mean(), 'Mean', C['gold']),
    (best_loss,         'Best', C['green']),
]:
    fig.add_vline(x=val, row=1, col=2,
                  line=dict(color=col, width=1.5, dash='dash'),
                  annotation_text=lbl,
                  annotation_font_color=col, annotation_font_size=10)

dark(fig, height=360)
fig.update_xaxes(title_text='Step',          row=1, col=1, title_font_color=C['text_dim'])
fig.update_xaxes(title_text='Loss value',    row=1, col=2, title_font_color=C['text_dim'])
fig.update_yaxes(title_text='Tokens / sec',  row=1, col=1, title_font_color=C['text_dim'])
fig.update_yaxes(title_text='Frequency',     row=1, col=2, title_font_color=C['text_dim'])
fig.update_annotations(font_size=12, font_color=C['text_dim'])
fig.show()

In [ ]:
# ── 8. Health heatmap (loss by epoch × step bucket) ──────────────────────
if 'epoch' in df.columns and df['epoch'].nunique() > 1:
    n_buckets = min(20, len(df))
    df['bucket'] = pd.cut(df['step'], bins=n_buckets, labels=False)
    heat = df.groupby(['epoch','bucket'])['loss'].mean().unstack(fill_value=np.nan)

    fig = go.Figure(go.Heatmap(
        z=heat.values,
        x=[f'Bucket {i}' for i in heat.columns],
        y=[f'Epoch {e}' for e in heat.index],
        colorscale=[[0.0, C['green']],
                    [0.5, C['gold']],
                    [1.0, C['red']]],
        colorbar=dict(title='Loss', tickfont=dict(color=C['text_dim']),
                      titlefont=dict(color=C['text_dim'])),
        hovertemplate='Epoch %{y}<br>%{x}<br>Loss: %{z:.4f}<extra></extra>',
    ))
    dark(fig, title='Loss Heatmap · Epoch × Step Buckets', height=300)
    fig.show()
else:
    # Single-epoch: show rolling-loss heatmap over time buckets
    n_buckets = min(25, len(df))
    df['bucket'] = pd.cut(df['step'], bins=n_buckets, labels=False)
    heat_row = df.groupby('bucket')['loss'].mean().values.reshape(1, -1)

    fig = go.Figure(go.Heatmap(
        z=heat_row,
        x=[f'S{i}' for i in range(heat_row.shape[1])],
        y=['Loss'],
        colorscale=[[0.0, C['green']],[0.5, C['gold']],[1.0, C['red']]],
        colorbar=dict(title='Loss', tickfont=dict(color=C['text_dim']),
                      titlefont=dict(color=C['text_dim'])),
        showscale=True,
        hovertemplate='Bucket %{x}<br>Avg Loss: %{z:.4f}<extra></extra>',
    ))
    dark(fig, title='Loss Heatmap · Step Buckets (single epoch)', height=180)
    fig.update_layout(margin=dict(t=56, b=24))
    fig.show()

In [ ]:
# ── 9. Correlation matrix ─────────────────────────────────────────────────
cols_for_corr = [c for c in ['loss','lr','vram_gb','tokens_per_sec','elapsed_sec']
                 if c in df.columns]
corr = df[cols_for_corr].corr()

labels = [c.replace('_',' ').title() for c in cols_for_corr]
fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=labels, y=labels,
    colorscale=[[0.0, C['red']],
                [0.5, C['surface2']],
                [1.0, C['blue']]],
    zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
    textfont=dict(color=C['text'], size=11),
    colorbar=dict(title='Pearson r', tickfont=dict(color=C['text_dim']),
                  titlefont=dict(color=C['text_dim'])),
    hovertemplate='%{y} × %{x}<br>r = %{z:.3f}<extra></extra>',
))
dark(fig, title='Metric Correlation Matrix', height=380)
fig.update_layout(xaxis=dict(tickangle=-30), margin=dict(b=80))
fig.show()

In [ ]:
# ── 10. Data table — styled for enterprise audit ──────────────────────────
display_cols = [c for c in ['step','epoch','loss','lr','vram_gb',
                             'tokens_per_sec','elapsed_sec'] if c in df.columns]
sample = df[display_cols].iloc[::max(1, len(df)//20)].copy()   # up to 20 rows

def fmt(val, col):
    if col == 'lr':           return f'{val:.2e}'
    if col in ('loss',):      return f'{val:.4f}'
    if col == 'vram_gb':      return f'{val:.2f}'
    if col == 'elapsed_sec':  return f'{val:.0f}'
    return str(int(val)) if pd.notna(val) else '—'

headers = ' '.join(f'<th>{h.replace("_"," ").upper()}</th>' for h in display_cols)
rows_html = ''
for _, row in sample.iterrows():
    loss_val = row.get('loss', 0)
    if loss_val == df['loss'].min():
        row_style = f'background:{C["surface2"]};border-left:3px solid {C["green"]};'
    else:
        row_style = ''
    cells = ' '.join(f'<td>{fmt(row[c], c)}</td>' for c in display_cols)
    rows_html += f'<tr style="{row_style}">{cells}</tr>'

table_html = f'''
<div style="background:{C["surface"]};border:1px solid {C["border"]};
            border-radius:8px;overflow:hidden;margin-top:8px;font-family:Inter,sans-serif;">
  <div style="padding:14px 20px;border-bottom:1px solid {C["border"]};
              display:flex;align-items:center;gap:12px;">
    <span style="font-size:13px;font-weight:600;color:{C["text"]}">Training Log</span>
    <span style="font-size:11px;color:{C["text_dim"]}">Sampled · {len(sample)} of {len(df)} rows · 
      <span style="color:{C["green"]}">■</span> Best checkpoint</span>
  </div>
  <div style="overflow-x:auto;">
  <table style="width:100%;border-collapse:collapse;
                font-size:12px;font-family:'JetBrains Mono',monospace;">
    <thead>
      <tr style="background:{C["surface2"]};color:{C["text_dim"]};
                 font-size:10px;letter-spacing:0.07em;">
        {headers}
      </tr>
    </thead>
    <tbody style="color:{C["text"]}">
      {rows_html}
    </tbody>
  </table>
  </div>
</div>
'''
display(HTML(table_html))

In [ ]:
# ── 11. Run summary card ──────────────────────────────────────────────────
initial_lr = df['lr'].max()
final_lr   = df['lr'].iloc[-1]

def row_html(label, value, accent=C['text']):
    return f'''
    <tr>
      <td style="color:{C["text_dim"]};padding:8px 16px;font-size:12px;
                 border-bottom:1px solid {C["border"]};width:220px;">{label}</td>
      <td style="color:{accent};padding:8px 16px;font-size:12px;
                 border-bottom:1px solid {C["border"]};
                 font-family:'JetBrains Mono',monospace;">{value}</td>
    </tr>'''

summary_rows = ''.join([
    row_html('Run name',          ACTIVE_RUN,                          C['blue']),
    row_html('Log file',          str(log_path),                       C['text_dim']),
    row_html('Steps logged',      f'{len(df)}',                        C['text']),
    row_html('Initial loss',      f"{df['loss'].iloc[0]:.4f}",         C['text']),
    row_html('Final loss',        f'{final_loss:.4f}',                  C['gold']),
    row_html('Best loss',         f'{best_loss:.4f}  (step {best_step})', C['green']),
    row_html('Loss reduction',    f'{pct_drop:.1f}%',                  C['green'] if pct_drop > 0 else C['red']),
    row_html('Peak VRAM',         f"{df['vram_gb'].max():.2f} GB / {VRAM_BUDGET:.0f} GB budget", C['text']),
    row_html('Avg throughput',    f"{df['tokens_per_sec'].mean():.0f} tok/s", C['text']),
    row_html('Peak throughput',   f"{df['tokens_per_sec'].max():.0f} tok/s", C['text']),
    row_html('Initial LR',        f'{initial_lr:.2e}',                 C['text']),
    row_html('Final LR',          f'{final_lr:.2e}',                   C['text']),
    row_html('Total training time', f"{int(total_sec//60)}m {int(total_sec%60)}s", C['purple']),
    row_html('Dashboard generated', run_date,                          C['text_dim']),
])

summary_html = f'''
<div style="background:{C["surface"]};border:1px solid {C["border"]};
            border-radius:8px;overflow:hidden;margin-top:8px;
            font-family:Inter,sans-serif;max-width:680px;">
  <div style="padding:14px 20px;border-bottom:1px solid {C["border"]}">
    <span style="font-size:13px;font-weight:600;color:{C["text"]}">Run Summary</span>
  </div>
  <table style="width:100%;border-collapse:collapse;">
    {summary_rows}
  </table>
</div>
'''
display(HTML(summary_html))

In [ ]:
# ── 12. Multi-run overlay (compare runs) ─────────────────────────────────
# Load all available run logs and overlay their smoothed loss curves.
available = {name: Path(p) for name, p in RUNS.items() if Path(p).exists()}

if len(available) > 1:
    PALETTE = [C['blue'], C['gold'], C['green'], C['purple'], C['orange']]
    fig = go.Figure()
    for (name, path), color in zip(available.items(), PALETTE):
        d = pd.read_csv(path)
        sg_w = min(len(d) if len(d)%2==1 else len(d)-1, 11)
        smooth = savgol_filter(d['loss'], sg_w, 2) if len(d) > 4 else d['loss']
        fig.add_trace(go.Scatter(
            x=d['step'], y=d['loss'],
            mode='lines', name=name + ' (raw)',
            line=dict(color=color, width=1),
            opacity=0.25, showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=d['step'], y=smooth,
            mode='lines', name=name,
            line=dict(color=color, width=2.5),
        ))
    dark(fig, title='Multi-Run Loss Comparison', height=380)
    fig.update_layout(xaxis_title='Step', yaxis_title='Loss', hovermode='x unified')
    fig.show()
else:
    display(HTML(
        f'<div style="background:{C["surface"]};border:1px solid {C["border"]};\''
        f'border-radius:8px;padding:20px 24px;color:{C["text_dim"]};font-size:13px;\''
        f'font-family:Inter,sans-serif;">'
        f'Multi-run comparison: add a second run log to <code>RUNS</code> in cell 2 to compare.'
        f'</div>'
    ))

In [ ]:
# ── 13. Export ────────────────────────────────────────────────────────────
# Static PNG of the loss curve (requires: pip install kaleido)
# Uncomment and run after a full training run.

# EXPORT_PATH = './outputs/dashboard_loss.png'
# fig_export = go.Figure()
# fig_export.add_trace(go.Scatter(
#     x=df['step'], y=df['loss_sg'], mode='lines',
#     line=dict(color=C['blue'], width=3)
# ))
# dark(fig_export, title=f'AetherForge · {ACTIVE_RUN}', height=500)
# pio.write_image(fig_export, EXPORT_PATH, width=1600, height=500, scale=2)
# print(f'Exported → {EXPORT_PATH}')